In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.io.arff import loadarff
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Conv1D,Conv2D, MaxPooling1D, Flatten, Dropout, Dense, LSTM,BatchNormalization

In [ ]:
fp= '../ScenarioA1/TimeBasedFeatures-Dataset-15s-VPN.arff'
raw_data = loadarff(fp)

In [ ]:
df_A1 = pd.DataFrame(raw_data[0])
df_A1

In [ ]:
df=df_A1
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
df['duration']=le.fit_transform(df.duration)
df['total_fiat']=le.fit_transform(df.total_fiat)
df['total_biat']=le.fit_transform(df.total_biat)
df['min_fiat']=le.fit_transform(df.min_fiat)
df['max_fiat']=le.fit_transform(df.max_fiat)
df['max_biat']=le.fit_transform(df.max_biat)
df['mean_fiat']=le.fit_transform(df.mean_fiat)
df['mean_biat']=le.fit_transform(df.mean_biat)
df['flowPktsPerSecond']=le.fit_transform(df.flowPktsPerSecond)
df['flowBytesPerSecond']=le.fit_transform(df.flowBytesPerSecond)
df['min_flowiat']=le.fit_transform(df.min_flowiat)
df['max_flowiat']=le.fit_transform(df.max_flowiat)
df['mean_flowiat']=le.fit_transform(df.mean_flowiat)
df['std_flowiat']=le.fit_transform(df.std_flowiat)
df['min_active']=le.fit_transform(df.min_active)
df['mean_active']=le.fit_transform(df.mean_active)
df['max_active']=le.fit_transform(df.max_active)
df['std_active']=le.fit_transform(df.std_active)
df['min_idle']=le.fit_transform(df.min_idle)
df['mean_idle']=le.fit_transform(df.mean_idle)
df['max_idle']=le.fit_transform(df.max_idle)
df['std_idle']=le.fit_transform(df.std_idle)
df_A1=df

In [ ]:
df_A1['class1'].value_counts()

In [ ]:
df['class1'] = df['class1'].map(lambda val : 0 if val == b'Non-VPN' else 1)

In [ ]:
X = df_A1.drop(columns=["class1"])
y = df_A1["class1"]

In [ ]:
random_state = 1
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=random_state, stratify=y)

In [ ]:
X_train.shape

In [ ]:
X=df.drop(columns=["class1"])
y=df["class1"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=1)
sc=StandardScaler()
X_train=sc.fit_transform(X_train)
X_test=sc.fit_transform(X_test)
print(X.shape)

In [ ]:
modelCNN = Sequential()
modelCNN.add(Conv1D(filters=64, kernel_size=5, input_shape=(23,1)))
modelCNN.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
modelCNN.add(BatchNormalization())
modelCNN.add(MaxPooling1D(pool_size=2))
modelCNN.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
modelCNN.add(Conv1D(filters=16, kernel_size=1, activation='relu'))
modelCNN.add(Flatten())
modelCNN.add(Dense(1000, activation='relu'))
modelCNN.add(Dense(256, activation='relu'))
modelCNN.add(Dropout(0.2))
modelCNN.add(Dense(2, activation='softmax'))

In [ ]:
modelRNN = Sequential()
modelRNN.add(LSTM(128,  activation='relu', input_shape=(23,1), return_sequences=True))
modelRNN.add(Dropout(0.2))
modelRNN.add(LSTM(64, activation='relu'))
modelRNN.add(Dropout(0.2))
modelRNN.add(Dense(32, activation='relu'))
modelRNN.add(Dropout(0.2))
modelRNN.add(Dense(2, activation='softmax'))

In [ ]:
modelDNN = Sequential()
modelDNN.add(Dense(32,input_dim=23,activation="relu"))
modelDNN.add(Dense(32, activation='relu'))
modelDNN.add(Dense(16, activation='relu'))
modelDNN.add(Dense(2, activation='sigmoid'))

In [ ]:
modelCNN.summary()
modelRNN.summary()
modelDNN.summary()

In [ ]:
import numpy as np
import pandas as pd
import gc
import random
from collections import OrderedDict
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn import preprocessing
from keras import backend as K
from keras.models import Sequential, load_model
from tensorflow.keras.optimizers import Adam, Nadam
from keras.initializers import glorot_uniform
from keras.layers import Dense, Conv1D, Flatten, MaxPool1D, Dropout, Activation, BatchNormalization
from keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import initializers
from keras.initializers import glorot_uniform


In [ ]:
modelCNN.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
modelCNN.fit(X_train, y_train, epochs=10)

In [ ]:
modelRNN.compile(loss='sparse_categorical_crossentropy',optimizer='adam',metrics=['accuracy'])
modelRNN.fit(X_train, y_train, epochs = 10)

In [ ]:
modelDNN.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
modelDNN.fit(X_train, y_train, epochs = 30)

In [ ]:
CNN_y_pred = modelCNN.predict(X_test)   
RNN_y_pred = modelRNN.predict(X_test)
DNN_y_pred = modelDNN.predict(X_test)

In [ ]:
CNN_y_pred = np.argmax(CNN_y_pred, axis=1)
RNN_y_pred = np.argmax(RNN_y_pred, axis=1)
DNN_y_pred = np.argmax(DNN_y_pred, axis=1)

In [ ]:
CNNResult = accuracy_score(y_test, CNN_y_pred)
RNNResult = accuracy_score(y_test, RNN_y_pred)
DNNResult = accuracy_score(y_test, DNN_y_pred)

In [ ]:
print('Scenario A1 CNN Accuracy Score: {0:0.4f}'.format(CNNResult))
print('Scenario A1 RNN Accuracy Score: {0:0.4f}'.format(RNNResult))
print('Scenario A1 DNN Accuracy Score: {0:0.4f}'.format(DNNResult))

In [ ]:
CNN_y_pred = modelCNN.predict(X_test) 
CNN_y_pred = np.argmax(CNN_y_pred, axis=1)
CNNResult = accuracy_score(y_test, CNN_y_pred)
print('Scenario A1 CNN Accuracy Score: {0:0.4f}'.format(CNNResult))

In [ ]:
CNN_cm = confusion_matrix(y_test, CNN_y_pred)
RNN_cm = confusion_matrix(y_test, RNN_y_pred)
DNN_cm = confusion_matrix(y_test, DNN_y_pred)

In [ ]:
target_names = ["b'VPN'","b'Non-VPN'"] 
CNN_cm_df = pd.DataFrame(CNN_cm,
                     index = target_names, 
                     columns = target_names)
RNN_cm_df = pd.DataFrame(RNN_cm,
                     index = target_names, 
                     columns = target_names)
DNN_cm_df = pd.DataFrame(DNN_cm,
                     index = target_names, 
                     columns = target_names)

In [ ]:
def createConfusionMatrix(matrix,title):
    plt.figure(figsize=(10,10))
    sns.heatmap(matrix, annot=True, fmt='g')
    plt.title(title, fontsize=20)
    plt.ylabel('Actual Values')
    plt.xlabel('Predicted Values')
    plt.show()

In [ ]:
CNN_title = "Scenario A1 CNN Model Confusion Matrix"
createConfusionMatrix(CNN_cm_df, CNN_title)

In [ ]:
RNN_title = "Scenario A1 RNN Model Confusion Matrix"
createConfusionMatrix(RNN_cm_df, RNN_title)

In [ ]:
DNN_title = "Scenario A1 DNN Model Confusion Matrix"
createConfusionMatrix(DNN_cm_df, DNN_title)

In [ ]:
CNN_report = classification_report(y_test, CNN_y_pred, target_names=target_names)
RNN_report = classification_report(y_test, RNN_y_pred, target_names=target_names)
DNN_report = classification_report(y_test, DNN_y_pred, target_names=target_names)

In [ ]:
print("     Scenario A1 CNN Classification Report\n", CNN_report)

In [ ]:
print("     Scenario A1 RNN Classification Report\n", RNN_report)

In [ ]:
print("     Scenario A1 DNN Classification Report\n", DNN_report)